# Quantization - compare FP32 with INT8, INT6, and INT4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/01_quantization.ipynb)

**Goal.** Take one trained FP32 model and quantize its weights to **8, 6, and 4 bits**.
The network architecture and trained FP32 weights stay fixed, so every comparison isolates
the effect of numerical precision.

The parts to focus on are:

1. **The quantization formula** - how a float maps to an integer and back.
2. **Quantization error** - how error grows as fewer levels are available.
3. **The compression/quality trade-off** - size, accuracy, and latency for FP32/INT8/INT6/INT4.

At the end the notebook reports a complete table and plots for all four variants. Weight
codes are bit-packed, including true 6-bit and 4-bit storage, so the size comparison is not
silently limited by an `int8` container.

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.
The FP32 baseline is trained once; INT8, INT6, and INT4 are post-training variants.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- knobs you can play with ---
EPOCHS     = 12      # epochs to train the FP32 baseline
BATCH_SIZE = 128
LR         = 0.05

# Explicit precisions used throughout the notebook.
N_BITS_8 = 8
N_BITS_6 = 6
N_BITS_4 = 4
BIT_WIDTHS = (N_BITS_8, N_BITS_6, N_BITS_4)

## 2. Data - MNIST

10 classes (handwritten digits 0-9) of 28x28 grayscale images. Small and quick to download.

In [ ]:
mean = (0.1307,)
std  = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2)

classes = train_set.classes
print(len(train_set), "train /", len(test_set), "test images")

## 3. The model

A plain VGG-style CNN. Nothing special - it is just the FP32 network we will compress.
Almost all of its size lives in the conv/linear **weights**, which is exactly what we will
quantize.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class Net(nn.Module):               # ~1.1M params
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 64),   conv_block(64, 64),   nn.MaxPool2d(2),   # 28 -> 14
            conv_block(64, 128), conv_block(128, 128), nn.MaxPool2d(2),   # 14 -> 7
            conv_block(128, 256),conv_block(256, 256), nn.MaxPool2d(2),   # 7  -> 3
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"params: {count_params(Net()):,}")

## 4. Train / eval / metric utilities

Plain training and evaluation loops, a single-image latency probe (batch size 1 on
**CPU**, the realistic edge setting), and an on-disk model-size probe.

In [ ]:
def train(model, epochs=EPOCHS, tag=""):
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        acc = evaluate(model)
        print(f"[{tag}] epoch {ep+1:02d}/{epochs}  test_acc={acc:.2f}%")
    return model

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    model.train()
    return 100.0 * correct / total

@torch.no_grad()
def latency_ms(model, n=100):
    """Average single-image inference latency on CPU (ms)."""
    model = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):            # warmup
        model(x)
    t0 = time.time()
    for _ in range(n):
        model(x)
    return (time.time() - t0) / n * 1000.0

def model_size_mb(state_dict):
    """On-disk size of a saved state dict (MB)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getbuffer().nbytes / 1e6

## 5. Train the FP32 baseline

Train the network normally. This is the **"before"** model - full 32-bit floats.

In [ ]:
model_fp32 = train(Net(), tag="fp32")
fp32_acc = evaluate(model_fp32)
print(f"\nFP32 baseline accuracy: {fp32_acc:.2f}%")

## 6. The quantization scheme & formula

This is the heart of the exercise. A real number $x$ is mapped to a signed integer $q$ by a
**scale** and an integer **zero-point**:

$$
q = \mathrm{clip}\left(\mathrm{round}\left(\frac{x}{\text{scale}}\right)
    + \text{zero\_point}, q_{\min}, q_{\max}\right),
\qquad
\hat{x} = \text{scale}\cdot(q - \text{zero\_point})
$$

For signed $b$-bit quantization, $q_{\min}=-2^{b-1}$ and $q_{\max}=2^{b-1}-1$.
INT8 therefore has 256 integer codes, INT6 has 64, and INT4 has only 16. Fewer codes use
less storage but force more FP32 values onto the same level, increasing quantization error.

Two choices define a scheme:

- **Symmetric vs. affine.** Symmetric quantization fixes `zero_point = 0` and uses
  $\text{scale}=\max(|x|)/q_{\max}$, which is natural for zero-centered weights. Affine
  quantization fits the observed $[\min,\max]$ and is often useful for skewed activations.
- **Per-tensor vs. per-channel.** Per-tensor uses one scale for the whole weight tensor.
  Per-channel uses one scale per output channel, reducing the influence of outlier channels.

The ideal weight-only compression ratios are $32/8=4\times$, $32/6=5.33\times$, and
$32/4=8\times$. Real files also contain FP32 BatchNorm values, scales, and metadata, so the
measured end-to-end ratios are slightly lower.

In [ ]:
def quantize(x, n_bits=8, symmetric=True, per_channel=False):
    """Float tensor -> (integer codes q, scale, zero_point). Channel axis = dim 0."""
    if not 2 <= n_bits <= 8:
        raise ValueError("This lab expects n_bits in [2, 8].")

    qmin = -(2 ** (n_bits - 1))
    qmax = 2 ** (n_bits - 1) - 1

    if per_channel:
        flat = x.flatten(1)
        xmin, xmax = flat.min(1).values, flat.max(1).values
    else:
        xmin, xmax = x.min(), x.max()

    if symmetric:
        # One symmetric range centered on zero. Clamp protects all-zero tensors.
        max_abs = torch.maximum(xmin.abs(), xmax.abs())
        scale = (max_abs / qmax).clamp(min=1e-8)
        zero_point = torch.zeros_like(scale)
    else:
        # Affine range uses the observed minimum and maximum.
        scale = ((xmax - xmin) / (qmax - qmin)).clamp(min=1e-8)
        zero_point = (qmin - torch.round(xmin / scale)).clamp(qmin, qmax)

    shape = [-1] + [1] * (x.dim() - 1)
    s = scale.view(shape) if per_channel else scale
    z = zero_point.view(shape) if per_channel else zero_point

    q = (torch.round(x / s) + z).clamp(qmin, qmax).to(torch.int32)
    return q, scale, zero_point


def dequantize(q, scale, zero_point, per_channel=False):
    if per_channel:
        shape = [-1] + [1] * (q.dim() - 1)
        scale, zero_point = scale.view(shape), zero_point.view(shape)
    return scale * (q - zero_point)


# Compare 8/6/4-bit quantization on the exact same trained weight tensor.
w = model_fp32.features[0][0].weight.data.cpu()

q8b, s8b, z8b = quantize(w, n_bits=N_BITS_8, symmetric=True, per_channel=True)
q6b, s6b, z6b = quantize(w, n_bits=N_BITS_6, symmetric=True, per_channel=True)
q4b, s4b, z4b = quantize(w, n_bits=N_BITS_4, symmetric=True, per_channel=True)

w8b = dequantize(q8b, s8b, z8b, per_channel=True)
w6b = dequantize(q6b, s6b, z6b, per_channel=True)
w4b = dequantize(q4b, s4b, z4b, per_channel=True)

err8b = F.mse_loss(w8b, w).item()
err6b = F.mse_loss(w6b, w).item()
err4b = F.mse_loss(w4b, w).item()

error_by_bits = {N_BITS_8: err8b, N_BITS_6: err6b, N_BITS_4: err4b}
codes_by_bits = {
    N_BITS_8: (q8b, s8b, z8b),
    N_BITS_6: (q6b, s6b, z6b),
    N_BITS_4: (q4b, s4b, z4b),
}

print(f"{'format':<8}{'code range':<17}{'levels':>8}{'first-layer MSE':>20}{'ideal compression':>20}")
print("-" * 73)
for bits in BIT_WIDTHS:
    q, _, _ = codes_by_bits[bits]
    print(
        f"INT{bits:<5}[{q.min().item():>4}, {q.max().item():>3}]"
        f"{2 ** bits:>10}{error_by_bits[bits]:>20.3e}{32 / bits:>19.2f}x"
    )

# Keep the original per-tensor/per-channel lesson, now at every requested precision.
print("\nPer-tensor vs. per-channel MSE on the first convolution:")
for bits in BIT_WIDTHS:
    q_pt, s_pt, z_pt = quantize(w, n_bits=bits, symmetric=True, per_channel=False)
    err_pt = F.mse_loss(dequantize(q_pt, s_pt, z_pt), w).item()
    print(f"INT{bits}: per-tensor={err_pt:.3e} | per-channel={error_by_bits[bits]:.3e}")

## 7. What reducing precision does to the weights

Every quantized value lies on a discrete grid. INT8 has a dense grid and usually tracks the
FP32 distribution closely. INT6 makes the grid more visible. INT4 has only 16 possible
codes, so many original values collapse onto the same reconstructed value.

The histograms below use the same first-convolution weights for a fair visual comparison.

In [ ]:
import matplotlib.pyplot as plt

weight_views = [
    ("FP32 original", w, "#4c4c4c"),
    (f"INT{N_BITS_8} dequantized", w8b, "#2a9d8f"),
    (f"INT{N_BITS_6} dequantized", w6b, "#457b9d"),
    (f"INT{N_BITS_4} dequantized", w4b, "#e76f51"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, (title, values, color) in zip(axes.flat, weight_views):
    ax.hist(values.flatten().numpy(), bins=80, color=color, alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel("weight value")
    ax.set_ylabel("count")
    ax.grid(axis="y", alpha=0.2)
plt.suptitle("The same FP32 weights represented at four precisions", y=1.01)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
bits_for_plot = list(BIT_WIDTHS)
errors_for_plot = [error_by_bits[b] for b in bits_for_plot]
ax.plot(bits_for_plot, errors_for_plot, marker="o", linewidth=2, color="#7b2cbf")
ax.set_yscale("log")
ax.invert_xaxis()
ax.set_xticks(bits_for_plot)
ax.set_xlabel("bits per weight (fewer bits ->)")
ax.set_ylabel("first-layer quantization MSE (log scale)")
ax.set_title("Quantization error rises as precision falls")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Post-training quantization: quantize every weight

Now apply symmetric per-channel quantization to every convolution and linear weight. This is
**post-training quantization (PTQ)**: all variants start from the same trained FP32 model and
require no retraining.

For each bit width the notebook builds two views:

- a **packed state dict**, where signed codes are densely packed into a byte stream. This
  represents actual 8/6/4-bit storage and is used for the on-disk size measurement;
- a **dequantized model**, where each weight is replaced by $\hat{x}$. This fake-quantized
  copy runs on ordinary PyTorch float kernels and is used to measure accuracy and latency.

The packed representation stores scales and shape metadata as well as weight codes. Biases,
BatchNorm parameters, and running statistics remain FP32.

In [ ]:
QUANTIZABLE = (nn.Conv2d, nn.Linear)


def pack_signed_codes(q, n_bits):
    """Densely pack signed n-bit integer codes into a uint8 tensor."""
    qmin = -(2 ** (n_bits - 1))
    unsigned = (q.detach().cpu().to(torch.int64).flatten() - qmin).numpy()
    shifts = np.arange(n_bits - 1, -1, -1, dtype=np.int64)
    bit_rows = ((unsigned[:, None] >> shifts) & 1).astype(np.uint8)
    packed_bytes = np.packbits(bit_rows.reshape(-1), bitorder="big")
    return torch.from_numpy(packed_bytes.copy())


def quantize_model(model, n_bits=8, per_channel=True):
    """Return (bit-packed state dict, fake-quantized model) for weight-only PTQ."""
    packed = {}
    deq = copy.deepcopy(model).cpu().eval()
    quantized_weight_keys = set()

    for name, mod in deq.named_modules():
        if not isinstance(mod, QUANTIZABLE):
            continue

        q, s, z = quantize(
            mod.weight.data,
            n_bits=n_bits,
            symmetric=True,
            per_channel=per_channel,
        )
        prefix = name + ".weight"
        packed[prefix + "_packed"] = pack_signed_codes(q, n_bits)
        packed[prefix + "_scale"] = s.clone()
        packed[prefix + "_zero_point"] = z.to(torch.int16)
        packed[prefix + "_shape"] = torch.tensor(q.shape, dtype=torch.int32)
        packed[prefix + "_n_bits"] = torch.tensor(n_bits, dtype=torch.int8)
        quantized_weight_keys.add(prefix)

        # Fake quantization: accuracy reflects the reconstructed low-precision weights.
        mod.weight.data = dequantize(q, s, z, per_channel=per_channel)

    # Keep non-quantized state (biases, BatchNorm parameters and running statistics) in FP32.
    for key, value in deq.state_dict().items():
        if key not in quantized_weight_keys:
            packed[key] = value.clone()

    return packed, deq


# Build all requested per-channel models and preserve explicit, easy-to-inspect variables.
packed_int8, model_int8 = quantize_model(model_fp32, n_bits=N_BITS_8, per_channel=True)
packed_int6, model_int6 = quantize_model(model_fp32, n_bits=N_BITS_6, per_channel=True)
packed_int4, model_int4 = quantize_model(model_fp32, n_bits=N_BITS_4, per_channel=True)

int8_acc = evaluate(model_int8.to(device))
int6_acc = evaluate(model_int6.to(device))
int4_acc = evaluate(model_int4.to(device))

# Also quantify how much per-channel scaling helps over one scale per tensor.
per_tensor_acc = {}
for bits in BIT_WIDTHS:
    _, model_pt = quantize_model(model_fp32, n_bits=bits, per_channel=False)
    per_tensor_acc[bits] = evaluate(model_pt.to(device))

per_channel_acc = {N_BITS_8: int8_acc, N_BITS_6: int6_acc, N_BITS_4: int4_acc}

print(f"{'model':<15}{'per-tensor':>14}{'per-channel':>15}{'PC vs PT':>12}")
print("-" * 56)
for bits in BIT_WIDTHS:
    pt, pc = per_tensor_acc[bits], per_channel_acc[bits]
    print(f"INT{bits:<12}{pt:>13.2f}%{pc:>14.2f}%{pc - pt:>+11.2f}pp")

## 9. Full comparison - size, accuracy, error, compression, and latency

The final experiment compares the FP32 baseline with all three per-channel PTQ variants.
The table includes measured packed size, end-to-end compression, test accuracy, accuracy
change from FP32, first-layer MSE, and CPU latency.

Remember that latency here uses fake-quantized FP32 execution. It measures whether the
reconstructed model changes runtime, but it does not claim an integer-kernel speedup.

In [ ]:
fp32_size = model_size_mb(model_fp32.state_dict())
fp32_latency = latency_ms(model_fp32)

quantized_specs = [
    ("INT8", N_BITS_8, packed_int8, model_int8, int8_acc, err8b),
    ("INT6", N_BITS_6, packed_int6, model_int6, int6_acc, err6b),
    ("INT4", N_BITS_4, packed_int4, model_int4, int4_acc, err4b),
]

results = [{
    "name": "FP32",
    "bits": 32,
    "size_mb": fp32_size,
    "compression": 1.0,
    "accuracy": fp32_acc,
    "delta_pp": 0.0,
    "mse": np.nan,
    "latency_ms": fp32_latency,
}]

for name, bits, packed, model, accuracy, mse in quantized_specs:
    size_mb = model_size_mb(packed)
    results.append({
        "name": name,
        "bits": bits,
        "size_mb": size_mb,
        "compression": fp32_size / size_mb,
        "accuracy": accuracy,
        "delta_pp": accuracy - fp32_acc,
        "mse": mse,
        "latency_ms": latency_ms(model),
    })

print(
    f"{'model':<8}{'bits':>6}{'size':>11}{'compress':>11}"
    f"{'accuracy':>12}{'vs FP32':>11}{'weight MSE':>14}{'CPU latency':>15}"
)
print("-" * 99)
for r in results:
    mse_text = "-" if np.isnan(r["mse"]) else f'{r["mse"]:.3e}'
    print(
        f'{r["name"]:<8}{r["bits"]:>6}{r["size_mb"]:>9.2f}MB'
        f'{r["compression"]:>10.2f}x{r["accuracy"]:>11.2f}%'
        f'{r["delta_pp"]:>+10.2f}pp{mse_text:>14}{r["latency_ms"]:>13.2f}ms'
    )

print("\nDirect comparison with the FP32 baseline:")
for r in results[1:]:
    print(
        f'- {r["name"]}: {r["compression"]:.2f}x smaller, '
        f'{r["delta_pp"]:+.2f}pp accuracy change, '
        f'{r["latency_ms"] / fp32_latency:.2f}x measured latency ratio.'
    )

# --- plots: every requested metric plus the size/accuracy Pareto view ---
names = [r["name"] for r in results]
colors = ["#4c4c4c", "#2a9d8f", "#457b9d", "#e76f51"]
x = np.arange(len(results))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

metric_specs = [
    ("size_mb", "Packed model size (MB)", "MB"),
    ("accuracy", "MNIST test accuracy", "%"),
    ("compression", "Compression vs. FP32", "x"),
    ("latency_ms", "CPU latency / image", "ms"),
]

for ax, (key, title, unit) in zip(axes.flat[:4], metric_specs):
    values = [r[key] for r in results]
    bars = ax.bar(x, values, color=colors)
    ax.set_xticks(x, names)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.2)
    if key == "accuracy":
        ax.set_ylim(max(0, min(values) - 2), min(100, max(values) + 1))
    for bar, value in zip(bars, values):
        label = f"{value:.2f}{unit}"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label,
                ha="center", va="bottom", fontsize=9)

ax = axes[1, 1]
quant_results = results[1:]
q_names = [r["name"] for r in quant_results]
q_errors = [r["mse"] for r in quant_results]
bars = ax.bar(q_names, q_errors, color=colors[1:])
ax.set_yscale("log")
ax.set_title("First-layer quantization MSE")
ax.grid(axis="y", alpha=0.2)
for bar, value in zip(bars, q_errors):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.2e}",
            ha="center", va="bottom", fontsize=9)

ax = axes[1, 2]
for r, color in zip(results, colors):
    ax.scatter(r["size_mb"], r["accuracy"], s=120, color=color, label=r["name"])
    ax.annotate(r["name"], (r["size_mb"], r["accuracy"]),
                xytext=(6, 6), textcoords="offset points")
ax.set_xlabel("packed size (MB) - lower is better")
ax.set_ylabel("accuracy (%) - higher is better")
ax.set_title("Size/accuracy trade-off")
ax.grid(True, alpha=0.25)

plt.suptitle("FP32 vs. INT8 vs. INT6 vs. INT4", fontsize=15)
plt.tight_layout()
plt.show()

# --- final evaluation: choose the smallest model within a clear accuracy-loss budget ---
accuracy_budget_pp = 0.50
eligible = [r for r in results[1:] if fp32_acc - r["accuracy"] <= accuracy_budget_pp]
best_balance = min(eligible, key=lambda r: r["size_mb"]) if eligible else max(
    results[1:], key=lambda r: r["accuracy"]
)
smallest = min(results[1:], key=lambda r: r["size_mb"])
most_accurate_quantized = max(results[1:], key=lambda r: r["accuracy"])

print("\nFINAL EVALUATION")
print("=" * 72)
print(f'FP32: accuracy reference at {fp32_acc:.2f}% and largest size ({fp32_size:.2f} MB).')
print(
    f'{most_accurate_quantized["name"]}: best quantized accuracy '
    f'({most_accurate_quantized["accuracy"]:.2f}%), '
    f'{most_accurate_quantized["compression"]:.2f}x compression.'
)
print(
    f'{smallest["name"]}: smallest model ({smallest["size_mb"]:.2f} MB, '
    f'{smallest["compression"]:.2f}x compression) with '
    f'{smallest["delta_pp"]:+.2f}pp accuracy change.'
)
if eligible:
    print(
        f'Best balance under a {accuracy_budget_pp:.2f}pp accuracy-loss budget: '
        f'{best_balance["name"]} ({best_balance["compression"]:.2f}x smaller, '
        f'{best_balance["delta_pp"]:+.2f}pp vs. FP32).'
    )
else:
    print(
        f'No quantized model stayed within the {accuracy_budget_pp:.2f}pp budget; '
        f'{best_balance["name"]} retained the highest quantized accuracy.'
    )
print(
    "Latency is expected to remain close to FP32 here because these models run "
    "dequantized weights through FP32 kernels. Integer kernels are required for "
    "a genuine inference speedup."
)

## 10. How to interpret the final trade-off

The final table and plots should make three effects visible:

- **FP32** is the quality reference, but it uses the most storage.
- **INT8** is normally the conservative deployment choice: strong compression with the
  smallest quantization error and usually negligible accuracy change.
- **INT6** is a middle point: more compression than INT8, with moderate additional error.
- **INT4** is the aggressive choice: the smallest representation and largest theoretical
  memory/bandwidth benefit, but it has the greatest risk of accuracy loss.

The notebook selects a practical winner using an explicit rule: choose the smallest model
whose accuracy loss is at most **0.50 percentage points**. This avoids declaring one format
universally best; the answer depends on the accuracy budget of the target application.

**Important latency note.** The storage result is real because integer codes are bit-packed.
The inference models are dequantized and use ordinary FP32 PyTorch kernels, so their measured
latencies should be similar. A real INT8/INT4 speedup requires hardware and a runtime with
matching integer kernels. Six-bit kernels are less commonly supported, even when INT6 gives
an attractive storage/accuracy point.

Useful follow-up experiments are per-tensor vs. per-channel quantization, affine activation
quantization, quantization-aware training for INT4, and integer-kernel deployment.